# Pixeltable × Ray Data Integration Demo

This notebook demonstrates `PixeltableDatasource` and `PixeltableDatasink` — a Ray Data integration
that lets you read Pixeltable tables as Ray Datasets, apply distributed transforms, and write results back.

**What we'll cover:**
1. Read a Pixeltable table as a Ray Dataset (with parallel tasks)
2. Filter and transform with native Ray operators
3. Write results back to Pixeltable via `PixeltableDatasink`
4. Round-trip: read → transform → write
5. The `read_pixeltable()` convenience function

In [1]:
import logging
import warnings

# Suppress Ray and Pixeltable INFO noise in notebook output
logging.getLogger('ray').setLevel(logging.ERROR)
logging.getLogger('pixeltable').setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

import pixeltable as pxt
from pixeltable.io.ray import PixeltableDatasource, PixeltableDatasink, read_pixeltable
import ray
import ray.data
import numpy as np

ray.init(ignore_reinit_error=True, logging_level=logging.ERROR, log_to_driver=False)

Python version:,3.10.13
Ray version:,2.54.1


## Setup: create a source table

In [2]:
# Clean slate
pxt.drop_dir('ray_demo', force=True, if_not_exists='ignore')
pxt.create_dir('ray_demo')

# Source table: 50 articles with id, title, score
articles = pxt.create_table('ray_demo.articles', {
    'id': pxt.Int,
    'title': pxt.String,
    'score': pxt.Float,
})
articles.insert([
    {'id': i, 'title': f'Article {i:02d}', 'score': round(i * 0.1, 1)}
    for i in range(50)
])
print(f'Source table has {articles.count()} rows')
articles.limit(5).collect()

id,title,score
0,Article 00,0.
1,Article 01,0.1
2,Article 02,0.2
3,Article 03,0.3
4,Article 04,0.4


## 1. Read as a Ray Dataset

`PixeltableDatasource` partitions the table by row ranges (LIMIT/OFFSET) across `override_num_blocks` parallel tasks.

In [3]:
ds = ray.data.read_datasource(
    PixeltableDatasource('ray_demo.articles'),
    override_num_blocks=4,
)

print('Schema :', ds.schema())
print('Rows   :', ds.count())
ds.show(5)

2026-04-09 19:46:26,074	INFO dataset.py:3670 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.


2026-04-09 19:46:26,081	INFO logging.py:392 -- Registered dataset logger for dataset dataset_1_0


2026-04-09 19:46:26,089	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2026-04-09_19-46-22_336544_70499/logs/ray-data


2026-04-09 19:46:26,090	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadPixeltable] -> LimitOperator[limit=5]


2026-04-09 19:46:26,092	WARNING resource_manager.py:141 -- ⚠️  Ray's object store is configured to use only 7.4% of available memory (2.0GiB out of 27.0GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.


...

2026-04-09 19:46:26,426	INFO logging_progress.py:231 -- limit=5: 0/1


2026-04-09 19:46:26,426	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store


2026-04-09 19:46:26,426	INFO logging_progress.py:192 -- ============================================


2026-04-09 19:46:27,634	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_1_0 execution finished in 1.54 seconds


{'id': 0, 'title': 'Article 00', 'score': 0.0}
{'id': 1, 'title': 'Article 01', 'score': 0.10000000149011612}
{'id': 2, 'title': 'Article 02', 'score': 0.20000000298023224}
{'id': 3, 'title': 'Article 03', 'score': 0.30000001192092896}
{'id': 4, 'title': 'Article 04', 'score': 0.4000000059604645}


## 2. Column projection

Pass `columns=` to read only specific columns — the query is pushed down to Pixeltable.

In [4]:
ds_proj = ray.data.read_datasource(
    PixeltableDatasource('ray_demo.articles', columns=['id', 'score']),
    override_num_blocks=4,
)

print('Schema (2 cols):', ds_proj.schema())
ds_proj.show(3)

2026-04-09 19:46:27,653	INFO logging.py:392 -- Registered dataset logger for dataset dataset_3_0


2026-04-09 19:46:27,654	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_3_0. Full logs are in /tmp/ray/session_2026-04-09_19-46-22_336544_70499/logs/ray-data


2026-04-09 19:46:27,655	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_3_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadPixeltable] -> LimitOperator[limit=3]


2026-04-09 19:46:27,668	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_3_0 =======


2026-04-09 19:46:27,668	INFO logging_progress.py:225 -- Total Progress: 0/?


...

2026-04-09 19:46:27,670	INFO logging_progress.py:231 -- limit=3: 0/1


2026-04-09 19:46:27,671	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store


2026-04-09 19:46:27,671	INFO logging_progress.py:192 -- ============================================


2026-04-09 19:46:27,719	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_3_0 execution finished in 0.06 seconds


Schema (2 cols): Column  Type
------  ----
id      int64
score   float
{'id': 0, 'score': 0.0}
{'id': 1, 'score': 0.10000000149011612}
{'id': 2, 'score': 0.20000000298023224}


## 3. Distributed transform + write back

Filter rows where `score >= 3.0`, double the score, and write to a new Pixeltable table.

In [5]:
# Target table must exist before writing
top_articles = pxt.create_table('ray_demo.top_articles', {
    'id': pxt.Int,
    'title': pxt.String,
    'score': pxt.Float,
})

# Ray transform pipeline
transformed = (
    ds
    .filter(lambda row: row['score'] >= 3.0)
    .map(lambda row: {**row, 'score': row['score'] * 2})
)

transformed.write_datasink(PixeltableDatasink('ray_demo.top_articles'))

result = top_articles.order_by(top_articles.id).collect()
print(f'Rows written: {len(result)}')
result

Rows written: 20


id,title,score
30,Article 30,6.
31,Article 31,6.2
32,Article 32,6.4
33,Article 33,6.6
34,Article 34,6.8
35,Article 35,7.
36,Article 36,7.2
37,Article 37,7.4
38,Article 38,7.6
39,Article 39,7.8


## 4. Round-trip: read → transform → write

In [6]:
# Embeddings table: fixed-shape array column
embed_src = pxt.create_table('ray_demo.embed_src', {
    'id': pxt.Int,
    'embedding': pxt.Array[(8,), pxt.Float],
})
embed_src.insert([
    {'id': i, 'embedding': np.random.rand(8).astype(np.float32)}
    for i in range(20)
])

embed_dst = pxt.create_table('ray_demo.embed_dst', {
    'id': pxt.Int,
    'embedding': pxt.Array[(8,), pxt.Float],
})

# Read, L2-normalise each vector in Ray workers, write back
def normalize(row):
    v = np.array(row['embedding'], dtype=np.float32)
    row['embedding'] = (v / (np.linalg.norm(v) + 1e-9)).tolist()
    return row

(
    ray.data.read_datasource(PixeltableDatasource('ray_demo.embed_src'), override_num_blocks=4)
    .map(normalize)
    .write_datasink(PixeltableDatasink('ray_demo.embed_dst'))
)

out = embed_dst.order_by(embed_dst.id).collect()
print(f'Normalised {len(out)} rows')
# Spot-check: norm ≈ 1.0
sample = np.array(out['embedding'][0])
print(f'L2 norm of first row: {np.linalg.norm(sample):.6f}  (should be ~1.0)')

Normalised 20 rows
L2 norm of first row: 1.000000  (should be ~1.0)


## 5. `read_pixeltable()` convenience function

A thin wrapper around `ray.data.read_datasource(PixeltableDatasource(...))` for ergonomic one-liners.

In [7]:
ds2 = read_pixeltable('ray_demo.articles', columns=['id', 'title'])

print('Schema:', ds2.schema())
print('Rows  :', ds2.count())
ds2.show(3)

2026-04-09 19:46:30,874	INFO logging.py:392 -- Registered dataset logger for dataset dataset_15_0


2026-04-09 19:46:30,876	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_15_0. Full logs are in /tmp/ray/session_2026-04-09_19-46-22_336544_70499/logs/ray-data


2026-04-09 19:46:30,876	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_15_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadPixeltable] -> LimitOperator[limit=3]


2026-04-09 19:46:30,900	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_15_0 =======


2026-04-09 19:46:30,901	INFO logging_progress.py:225 -- Total Progress: 0/?


...

2026-04-09 19:46:30,903	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store


2026-04-09 19:46:30,904	INFO logging_progress.py:192 -- =============================================


Schema: Column  Type
------  ----
id      int64
title   string
Rows  : 50


2026-04-09 19:46:31,233	INFO streaming_executor.py:300 -- ✔️  Dataset dataset_15_0 execution finished in 0.30 seconds


{'id': 0, 'title': 'Article 00'}
{'id': 1, 'title': 'Article 01'}
{'id': 2, 'title': 'Article 02'}


## 6. Ray Train: Pixeltable as a feature store

Train a simple linear regression model distributed across workers.  
The dataset lives in Pixeltable; Ray Train reads it via `PixeltableDatasource` inside each worker.

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from ray import train
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer

# ── Build a synthetic regression dataset in Pixeltable ──────────────────
# y = 3*x1 - 2*x2 + noise  (2 features, 1 target)
np.random.seed(42)
N = 200
X = np.random.randn(N, 2).astype(np.float32)
y = (3 * X[:, 0] - 2 * X[:, 1] + 0.1 * np.random.randn(N)).astype(np.float32)

train_tbl = pxt.create_table('ray_demo.train_data', {
    'x1': pxt.Float,
    'x2': pxt.Float,
    'label': pxt.Float,
})
train_tbl.insert([{'x1': float(X[i, 0]), 'x2': float(X[i, 1]), 'label': float(y[i])} for i in range(N)])
print(f'Training table: {train_tbl.count()} rows, columns: x1, x2, label')
train_tbl.limit(3).collect()

x1,x2,label
0.497,-0.138,1.607
0.648,1.523,-1.163
-0.234,-0.234,-0.234


In [9]:
import os
import shutil
import tempfile

def train_loop(config: dict) -> None:
    """Runs on each Ray Train worker. Uses Ray Data streaming — no pandas conversion."""
    import os as _os, tempfile as _tmp
    import numpy as np
    import torch
    import torch.nn as nn
    from ray import train
    from ray.train import Checkpoint

    shard = train.get_dataset_shard('train')

    model     = train.torch.prepare_model(nn.Linear(2, 1))
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
    loss_fn   = nn.MSELoss()

    for epoch in range(30):
        model.train()
        for batch in shard.iter_batches(batch_size=32, batch_format='numpy'):
            X_b = torch.tensor(
                np.column_stack([batch['x1'], batch['x2']]), dtype=torch.float32
            )
            y_b = torch.tensor(batch['label'], dtype=torch.float32).unsqueeze(1)
            optimizer.zero_grad()
            loss_fn(model(X_b), y_b).backward()
            optimizer.step()

        with torch.no_grad():
            parts = [
                (torch.tensor(np.column_stack([b['x1'], b['x2']]), dtype=torch.float32),
                 torch.tensor(b['label'], dtype=torch.float32).unsqueeze(1))
                for b in shard.iter_batches(batch_size=64, batch_format='numpy')
            ]
            X_all = torch.cat([p[0] for p in parts])
            y_all = torch.cat([p[1] for p in parts])
            val_loss = loss_fn(model(X_all), y_all).item()

        inner = model.module if hasattr(model, 'module') else model
        with _tmp.TemporaryDirectory() as tmpdir:
            torch.save(inner.state_dict(), _os.path.join(tmpdir, 'model.pt'))
            train.report(
                {'epoch': epoch + 1, 'mse': round(val_loss, 5)},
                checkpoint=Checkpoint.from_directory(tmpdir),
            )


# Build dataset on the driver; Ray shards it across workers automatically
ds = ray.data.read_datasource(
    PixeltableDatasource('ray_demo.train_data', columns=['x1', 'x2', 'label']),
    override_num_blocks=4,
)

trainer = TorchTrainer(
    train_loop,
    datasets={'train': ds},
    scaling_config=ScalingConfig(num_workers=2, use_gpu=False),
)
result = trainer.fit()

ckpt_dir = result.checkpoint.to_directory()
state_dict = torch.load(os.path.join(ckpt_dir, 'model.pt'), weights_only=True)
w = [round(v, 3) for v in state_dict['weight'].squeeze().tolist()]
b = round(state_dict['bias'].item(), 3)

print(f'\nFinal MSE                              : {result.metrics["mse"]}')
print(f'Learned weights (expect ≈ [3.0, -2.0]): {w}')
print(f'Learned bias   (expect ≈  0.0)         : {b}')


Final MSE                              : 0.01146
Learned weights (expect ≈ [3.0, -2.0]): [3.007, -1.985]
Learned bias   (expect ≈  0.0)         : -0.022


## Cleanup

In [10]:
pxt.drop_dir('ray_demo', force=True)
ray.shutdown()
print('Done.')

Done.
